In [2]:
import pandas as pd
import polars as pl
import os
import numpy as np

In [3]:
# Load the dataset
df = pd.read_csv('data/euro_qualifying_1960-2024.csv')

In [4]:
df.head()

,id_match,home_team,away_team,home_team_code,away_team_code,home_score,away_score,home_penalty,away_penalty,home_score_total,...,penalties_missed,penalties,red_cards,game_referees,stadium_city,stadium_name,stadium_name_media,stadium_name_official,stadium_name_event,stadium_name_sponsor
0,3999,USSR,Hungary,URS,HUN,3.0,1.0,NaN,NaN,3.0,...,NaN,NaN,NaN,[],Moscow,Luzhniki Stadium,Luzhniki Stadium,Luzhniki Stadium,Luzhniki Stadium,Luzhniki Stadium
1,4000,France,Greece,FRA,GRE,7.0,1.0,NaN,NaN,7.0,...,NaN,NaN,NaN,[],Paris,Parc des Princes,Parc des Princes,Parc des Princes,Parc des Princes,Parc des Princes
2,4001,Romania,Türki̇ye,ROU,TUR,3.0,0.0,NaN,NaN,3.0,...,NaN,NaN,NaN,[],Bucharest,23 August,23 August,23 August,23 August,23 August
3,4002,Greece,France,GRE,FRA,1.0,1.0,NaN,NaN,1.0,...,NaN,NaN,NaN,[],Athens,Olympic Athletic Center of Athens,"OACA ""Spyros Louis""",Olympic Athletic Center of Athens,"Olympic Athletic Center of Athens ""Spyros Louis""","Olympic Athletic Center of Athens ""Spyros Louis"""
4,3997,Republic of Ireland,Czechoslovakia,IRL,TCH,2.0,0.0,NaN,NaN,2.0,...,NaN,NaN,NaN,[],Dublin,Dalymount Park,Dalymount Park,Dalymount Park,Dalymount Park,Dalymount Park


In [5]:
# Check dataset size
print("Shape of dataset:", df.shape)

# See column names
df.columns

Shape of dataset: (2845, 47)


Index(['id_match', 'home_team', 'away_team', 'home_team_code',
       'away_team_code', 'home_score', 'away_score', 'home_penalty',
       'away_penalty', 'home_score_total', 'away_score_total', 'winner',
       'winner_reason', 'year', 'date', 'date_time', 'utc_offset_hours',
       'group_name', 'matchday_name', 'condition_humidity', 'condition_pitch',
       'condition_temperature', 'condition_weather', 'condition_wind_speed',
       'status', 'type', 'round', 'round_mode', 'match_attendance',
       'stadium_id', 'stadium_country_code', 'stadium_capacity',
       'stadium_latitude', 'stadium_longitude', 'stadium_pitch_length',
       'stadium_pitch_width', 'goals', 'penalties_missed', 'penalties',
       'red_cards', 'game_referees', 'stadium_city', 'stadium_name',
       'stadium_name_media', 'stadium_name_official', 'stadium_name_event',
       'stadium_name_sponsor'],
      dtype='object')

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2845 entries, 0 to 2844
Data columns (total 47 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   id_match               2845 non-null   int64  
 1   home_team              2845 non-null   object 
 2   away_team              2845 non-null   object 
 3   home_team_code         2845 non-null   object 
 4   away_team_code         2845 non-null   object 
 5   home_score             2842 non-null   float64
 6   away_score             2842 non-null   float64
 7   home_penalty           7 non-null      float64
 8   away_penalty           7 non-null      float64
 9   home_score_total       2842 non-null   float64
 10  away_score_total       2842 non-null   float64
 11  winner                 2291 non-null   object 
 12  winner_reason          2842 non-null   object 
 13  year                   2845 non-null   int64  
 14  date                   2845 non-null   object 
 15  date

In [7]:
# Targated variable check
df["winner"].value_counts()

winner
Spain                    97
Netherlands              83
England                  79
Italy                    78
Portugal                 76
                         ..
Liechtenstein             5
Kosovo                    5
Malta                     4
Serbia and Montenegro     2
Andorra                   1
Name: count, Length: 61, dtype: int64

In [8]:
# Create match_result column using scores
df["match_result"] = df.apply(
    lambda row: "H" if row["home_score"] > row["away_score"]
    else "A" if row["home_score"] < row["away_score"]
    else "D",
    axis=1
)

df["match_result"].value_counts()

match_result
H    1377
A     902
D     566
Name: count, dtype: int64

In [9]:
# Check missing values in each column
missing_values = df.isnull().sum().sort_values(ascending=False)

missing_values

penalties                2839
home_penalty             2838
away_penalty             2838
penalties_missed         2797
red_cards                2506
condition_humidity       2477
condition_pitch          2365
condition_weather        2181
condition_temperature    1854
condition_wind_speed     1854
winner                    554
goals                     207
group_name                159
stadium_pitch_width        93
stadium_pitch_length       93
date_time                  83
utc_offset_hours           83
stadium_name_sponsor       41
match_attendance           23
stadium_latitude           14
stadium_longitude          14
stadium_name               11
stadium_name_media          4
stadium_city                4
stadium_name_official       4
stadium_country_code        4
stadium_capacity            4
stadium_id                  4
stadium_name_event          4
away_score                  3
home_score                  3
home_score_total            3
away_score_total            3
winner_rea

In [10]:
# remove rows where scores are missing
df = df.dropna(subset=["home_score", "away_score"]).copy()

# create target column
df["match_result"] = "D"
df.loc[df["home_score"] > df["away_score"], "match_result"] = "H"
df.loc[df["home_score"] < df["away_score"], "match_result"] = "A"

# convert date column
df["date"] = pd.to_datetime(df["date"], errors="coerce")

# if year has any issues, rebuild it from date when possible
df["year"] = df["date"].dt.year.fillna(df["year"]).astype("Int64")

# create decade column for later analysis
df["decade"] = (df["year"] // 10) * 10

# keep only important columns
keep_cols = [
    "date",
    "year",
    "decade",
    "home_team",
    "away_team",
    "home_team_code",
    "away_team_code",
    "home_score",
    "away_score",
    "match_result",
    "matchday_name",
    "round",
    "type",
    "match_attendance",
    "stadium_country_code",
    "stadium_capacity",
    "condition_temperature",
    "condition_weather",
    "condition_humidity",
    "condition_wind_speed",
    "condition_pitch"
]

df_clean = df[keep_cols].copy()

# check missing values in the kept columns
print(df_clean.isnull().sum().sort_values(ascending=False))
print("\nShape before final cleaning:", df_clean.shape)

condition_humidity       2474
condition_pitch          2362
condition_weather        2178
condition_temperature    1851
condition_wind_speed     1851
match_attendance           21
stadium_country_code        2
stadium_capacity            2
away_team                   0
date                        0
home_team                   0
decade                      0
year                        0
type                        0
round                       0
matchday_name               0
match_result                0
away_score                  0
home_score                  0
away_team_code              0
home_team_code              0
dtype: int64

Shape before final cleaning: (2842, 21)


In [11]:
# Columns with too many missing values
drop_cols = [
    "condition_humidity",
    "condition_pitch",
    "condition_weather",
    "condition_temperature",
    "condition_wind_speed"
]

df_clean = df_clean.drop(columns=drop_cols)

In [12]:
df_clean.head()

,date,year,decade,home_team,away_team,home_team_code,away_team_code,home_score,away_score,match_result,matchday_name,round,type,match_attendance,stadium_country_code,stadium_capacity
0,1958-09-28,1958,1950,USSR,Hungary,URS,HUN,3.0,1.0,H,1,ROUND_OF_16,FIRST_LEG,100572.0,RUS,81015.0
1,1958-10-01,1958,1950,France,Greece,FRA,GRE,7.0,1.0,H,1,ROUND_OF_16,FIRST_LEG,37590.0,FRA,47926.0
2,1958-11-02,1958,1950,Romania,Türki̇ye,ROU,TUR,3.0,0.0,H,1,ROUND_OF_16,FIRST_LEG,67200.0,ROU,0.0
3,1958-12-03,1958,1950,Greece,France,GRE,FRA,1.0,1.0,D,2,ROUND_OF_16,SECOND_LEG,18833.0,GRE,68224.0
4,1959-04-05,1959,1950,Republic of Ireland,Czechoslovakia,IRL,TCH,2.0,0.0,H,1,PRELIMINARY,FIRST_LEG,37500.0,IRL,2740.0
